<a href="https://colab.research.google.com/github/chamudithamk/Data-Driven-Dynamics-and-Control/blob/main/0_Data_Driven_Dynamical_Systems.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Data-Driven Dynamical Systems**

### **1. Overview of Dynamical Systems**

Traditionally, the dynamics of a system has been derived using Newtonian, Lagrangian or Hamiltonian first principles.

General form of a dynamical equation:

$$\dot{x} = f(x,t,u; \beta) + d$$

where $d$ is disturbance and $\beta$ is a system paramter that changes dynamic behaviour of a system (e.g., altitude of a aeroplane).

***NOTE**: $\beta$ can be used as a design parameter to control a system whereas $u$ is typically used in real-time feedback controllers*

However, first princples are only models of reality, sometimes they are extremely good and sometimes, $f$ is simply unknown. This is where data-driven approaches come into play. Data-driven approaches can tackle a number of other complexities as well: nonlinearity, uncertainty, high-dimensionality, stochasticity, chaos (highly sensitive to initial conditions and other parameters), transients, multi-scale (e.g., turbulence scales).

The measurements of such a dynamical system:

$$y = g(x,t) + n$$

where $n$ is noise. $y$ is not necessarily full-state and is estimated through an observer (e.g., Kalman Filter).

Ultimately, they can be used to predict future states, control and for design and optimisation purposes. Also, it is important that ML models are generalisable and interpretable rather than black-box models.

#### 1.1 Example: Simulation of Lorenz Equations

It is a set of three coupled nonlinear ordinary differential equations founded by Edward Lorenz in 1963 to show why long-range weather forecasting is fundamentally limited. It is the founding example of chaos theory and is a standard test problem for numerical solvers, data assimilation and ML time-series forecasting.

$$\dot{x} = \sigma(y-x)$$
$$\dot{y} = x(\rho -z)-y$$
$$\dot{z} = xy-\beta z$$

State-space form of this can be given by:

$$\dot{\mathbf{x}} = f(\mathbf{x},t; \mathbf{b})$$

where

$$\mathbf{x} = \begin{bmatrix}
x \\ y \\ z
\end{bmatrix}; \;\;\;\; \mathbf{b} = \begin{bmatrix} \sigma \\ \rho \\ \beta \end{bmatrix}$$



In [ ]:
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from scipy.integrate import solve_ivp

# ---------------- INPUTS ----------------
b  = [10.0, 28.0, 8/3]     # [sigma, rho, beta]
x0 = [1.0, 1.0, 1.0]       # initial state [x, y, z]
dt = 0.01                  # output time step
T  = 50.0                  # total duration
method = "RK45"            # "RK45", "DOP853", "LSODA", "Radau", "BDF"
# ----------------------------------------

In [ ]:
def lorenz(t, state, sigma, rho, beta):
    x, y, z = state
    return [sigma * (y - x),
            x * (rho - z) - y,
            x * y - beta * z]

t_eval = np.arange(0, T + dt / 2, dt)

sol = solve_ivp(lorenz, (0, T), x0, args=tuple(b),
                method=method, t_eval=t_eval,
                rtol=1e-9, atol=1e-12)
t = sol.t
x, y, z = sol.y

# --------- 3D Lorenz butterfly attractor visualisation ---------
fig3d = go.Figure(go.Scatter3d(
    x=x, y=y, z=z, mode="lines",
    line=dict(width=3, color=t, colorscale="Viridis",
              colorbar=dict(title="t"))
))
fig3d.update_layout(
    title=f"Lorenz attractor",
    scene=dict(xaxis_title="x", yaxis_title="y", zaxis_title="z"),
    height=650
)
fig3d.show()
# ----------------------------------------------------------------

**Q1a: What is Chaos?**
Small differences in initial conditions can grow until the resulting trajectories become significantly different.

**Q1b: Why does a continous-time system need at least 3-dimensions for Chaos to occur?**

The fundamental rule is that two trajectories cannot cross, if it does, it means it has the same initial condition at that point. In 1D, the state cannot reverse direction arbitrarily (it either converges or diverges); in 2D, the Poincaré–Bendixson theorem (if a state is in a bounded region, it either converges or in a limit cycle) rules out chaotic attractors. At least 3 state dimensions are needed for this kind of continuous-time chaos, but how?

Suppose two points start close together. As the system evolves, stretching causes the points to move farther apart, making the system sensitive to initial conditions. Meanwhile, the deterministic dynamics of the system cause the space to fold back on itself, bringing different regions into the same region. When stretching and folding occur repeatedly, they can produce chaotic dynamics, as observed in the Lorenz (butterfly) attractor.



#### 1.2 Discrete-Time Systems

Dynamic equation for a continous autonomous system:

$$\dot{x} = f(x)$$

A discrete-time system, however, evolves in steps.

$$x_{k+1} = F(x_k)$$

If $\Delta t$ is the time-step of an integrator, then the discrete-time dynamical system can be defined by:

$$x_k = x(k \cdot \Delta t)$$

Then, the flow map $F_{\Delta t}$ of states is given by:

$$x_{k+1} = x_k + \int_{k\Delta t}^{(k+1)\Delta t} f(x(\tau)) dt$$

In most dynamical systems of interest, computation of flow map $F_{\Delta t}$ solution in closed-form is not possible and must be adapted to certain numerical schemes such as Forward Euler, Runge-Kutta, etc.

#### 1.3 Example: Logistic Map Discrete-Time System

It is a population dynamic model popularised by biologist Robert May in 1976, which shows trivially simple nonlinear rule can produce extremely complicated behaviour.

$$x_{k+1}=\beta x_k (1-x_k)$$

where $\beta$ is the growth rate of population $x_k$. Due to finite Earth resources, term $(1-x_k)$ is introduced as the crowding that limits the growth.

Let us demonstrate this system's period-doubling route to chaos as $\beta$ increases through its bifurcation diagram.

In [ ]:
betas = np.linspace(0, 4.0, 500)
n_transient, n_keep = 500, 200

x = 0.5 * np.ones_like(betas)
for _ in range(n_transient):
    x = betas * x * (1 - x)

B, X = [], []
for _ in range(n_keep):
    x = betas * x * (1 - x)
    B.append(betas.copy()); X.append(x.copy())

fig = go.Figure(go.Scattergl(
    x=np.concatenate(B), y=np.concatenate(X),
    mode="markers", marker=dict(size=1.5, color="black", opacity=0.3)))
fig.update_layout(title="Logistic map bifurcation diagram",
                  xaxis_title="β", yaxis_title="x (steady state)", height=600)
fig.show()

From the above bifurcation diagram, it can be seen that:

<center>
<table border="1">
  <tr align="center">
    <td><b>Parameter Range</b></td>
    <td><b>Population Behavior</b></td>
    <td><b>Dynamic State</b></td>
  </tr>

  <tr align="center">
    <td>&beta; &le; 1</td>
    <td>Population dies out i.e. x &rarr; 0</td>
    <td>Extinction</td>
  </tr>
  <tr align="center">
    <td>1 &lt; &beta; &le; 3</td>
    <td>Population stabilises to a finite value</td>
    <td>Stable Equilibrium</td>
  </tr>
  <tr align="center">
    <td>3 &lt; &beta; &le; 3.45</td>
    <td>Population oscillates between two values after first bifurcation</td>
    <td>Period-2 Orbit</td>
  </tr>
  <tr align="center">
    <td>3.45 &lt; &beta; &le; 3.54</td>
    <td>Population oscillates between four values after second bifurcation</td>
    <td>Period-4 Orbit</td>
  </tr>
  <tr align="center">
    <td>3.54 &lt; &beta; &le; 3.84</td>
    <td>Chaotic population wanders without repeating</td>
    <td>Chaos</td>
  </tr>
  <tr align="center">
    <td>&beta; &approx; 3.84</td>
    <td>Population oscillates between three values</td>
    <td>Order Inside Chaos</td>
  </tr>
  <tr align="center">
    <td>3.84 &lt; &beta; &le; 4</td>
    <td>Fully developed chaos</td>
    <td>Complete Chaos</td>
  </tr>
  <tr align="center">
    <td>&beta; &gt; 4</td>
    <td>Population escapes [0,1]</td>
    <td>Divergence</td>
  </tr>
</table>
</center>

**Q1c: How can a 1-dimensional discrete-time system turn into chaos?**

In discrete systems, chaos can occur even with 1 state variable, as in the logistic map. Here, due to the boundedness $[0,1]$ of the dynamic equation of logistic map, and the ability to jump between states $x_k$ and $x_{k+1}$,


*   Two nearby initial conditions can stretch apart, as in continuous-time systems.
*   The map can also fold, mapping different states back into the same region and reversing their order without necessarily revisiting the same state.

When stretching and folding occur repeatedly, chaotic behaviour can emerge. In a one-dimensional autonomous continuous-time system, trajectories cannot reverse direction or cross one another, since doing so would violate uniqueness of solutions.



### **2. Dynamic Mode Decomposition (DMD)**

Dynamic Mode Decomposition (DMD) is a data-driven (no governing equations required) modal decomposition where each mode consists of spatially correlated structures that have the same linear
behavior in time (e.g., oscillations at a given frequency with growth or decay).

First, a sufficient number ($m$) of snapshots of data (states) must be collected $\{ \mathbf{x}_1, \mathbf{x}_2, \dots, \mathbf{x}_m \} \in \mathbb{R}^{\text{n}}$ as the dynamics evolve in time. Time-steps $\Delta t$ must be sufficiently small to resolve the highest frequencies in the dynamics.

Next, organise them into matrices $\mathbf{X}, \mathbf{X} \in \mathbb{M}^{n \times (m-1)}$ such that:

$$\mathbf{X} = \begin{bmatrix}
  \mid & \mid & & \mid & \mid \\
  \mathbf{x}(t_1) & \mathbf{x}(t_2) & \dots & \mathbf{x}(t_{m-2}) & \mathbf{x}(t_{m-1})  \\
  \mid & \mid & & \mid & \mid
\end{bmatrix}$$

$$\mathbf{X}' = \begin{bmatrix}
  \mid & \mid & & \mid & \mid \\
  \mathbf{x}(t_2) & \mathbf{x}(t_3) & \dots & \mathbf{x}(t_{m-1}) & \mathbf{x}(t_{m})  \\
  \mid & \mid & & \mid & \mid
\end{bmatrix}$$

DMD algorithm then seeks a best-fit linear operator $\mathbf{A}$ (regression) that advances $\mathbf{X}$ to $\mathbf{X}'$ in time.

$$\mathbf{X}' \approx \mathbf{A} \mathbf{X}$$

If we assume uniform sampling in time,

$$\mathbf{x}_k = \mathbf{x}(k \cdot \Delta t)$$
$$\mathbf{x}_{k+1} \approx \mathbf{A}\mathbf{x}_k$$

Mathematically, best-fit linear operator $\mathbf{A}$ is:
$$\mathbf{A} = \mathbf{X}'\mathbf{X}^\dagger$$

We are often interested in eigenvectors $\Phi$ and eigenvalues $\Lambda$ of $\mathbf{A}$.

$$\mathbf{A}\Phi = \Phi \Lambda $$

However, for a high-dimensional state vector $\mathbf{x} \in \mathbb{R}^{\text{n}}$, the matrix $\mathbf{A}$ has $n^2$ elements, and representing this operator, let alone computing its spectral decomposition, may be intractable.

Instead, the DMD algorithm leverages dimensionality reduction (SVD) to compute the dominant eigenvalues and eigenvectors of $\mathbf{A}$ without requiring any explicit computations using $\mathbf{A}$ directly.

Since typically $m \ll n$, DMD algorithm projects $\mathbf{A}$, which has rank $m$ at most, onto its leading $m$ singular vectors (dominant modes). This results in a small matrix $\tilde{\mathbf{A}} \in \mathbb{M}^{m\times m}$ with $m^2$ elements.

***NOTE:** Each DMD mode is associated with a particular eigenvalue $\lambda = a + ib$ where $b$ is the frequency of oscillation and $a$ is the growth or decay rate.*

#### 2.1 DMD Algorithm

First, compute the singular value decomposition (SVD) of $\mathbf{X}$:

$$\mathbf{X} \approx \tilde{\mathbf{U}}\tilde{\Sigma}\tilde{\mathbf{V}}^*$$

where $\tilde{\mathbf{U}} \in \mathbb{C}^{n \times r}$, $\tilde{\Sigma} \in \mathbb{R}^{r \times r}$, $\tilde{\mathbf{V}} \in \mathbb{C}^{(m-1) \times r}$ and $r \le m$. Also $\tilde{\mathbf{U}}^*\tilde{\mathbf{U}} = \mathbf{I}$ and $\tilde{\mathbf{V}}^*\tilde{\mathbf{V}} = \mathbf{I}$

In practice, choosing the approximate rank $r$ of $\mathbf{X}$ is one of the most important and subjective steps in DMD [Q2a]. For example, if first $r' (\ll r)$ POD modes [Q2b] capture 99% of the dynamics the system, $r'$ is optimal.

Second, $\mathbf{A}$ may be obtained through:

$$\mathbf{A} = \mathbf{X}'\mathbf{X}^\dagger = \mathbf{X}'\tilde{\mathbf{V}}\tilde{\Sigma}^{-1}\tilde{\mathbf{U}}^*$$

However, we are only interested in the leading $r$ eigenvalues and eigenvectors of $\mathbf{A}$, and we may thus project $\mathbf{A}$ onto the POD modes in $\mathbf{U}$.

$$\tilde{\mathbf{A}} = \tilde{\mathbf{U}}^*\mathbf{A}\tilde{\mathbf{U}} = \tilde{\mathbf{U}}^*(\mathbf{X}'\tilde{\mathbf{V}}\tilde{\Sigma}^{-1}\tilde{\mathbf{U}}^*)\tilde{\mathbf{U}} = \tilde{\mathbf{U}}^*\mathbf{X}'\tilde{\mathbf{V}}\tilde{\Sigma}^{-1} $$

The reduced-order matrix $\tilde{\mathbf{A}}$ defines a linear model for the dynamics of the vector of POD coefficients $\tilde{\mathbf{x}}$:

$$\tilde{\mathbf{x}}_{k+1} = \tilde{\mathbf{A}}\tilde{\mathbf{x}}_k$$

where the map between reduced state and full state is given by: $\mathbf{x} = \tilde{\mathbf{U}}\tilde{\mathbf{x}}$

Note that the reduced matrix $\tilde{\mathbf{A}}$ has the same nonzero eigenvalues as the full matrix $\mathbf{A}$ [Q2c]. Thus, we only need to compute $\tilde{\mathbf{A}}$ directly.

Third, the spectral decomposition of $\tilde{\mathbf{A}}$ is computed.

$$\tilde{\mathbf{A}}\mathbf{W} = \mathbf{W}\Lambda$$

Columns of $\mathbf{W}$ (eigenvectors of $\tilde{\mathbf{A}}$) may be thought of as linear combinations of POD mode amplitudes that behave linearly with a single temporal pattern given by $\lambda$.

Finally, the high-dimensional DMD modes $\Phi$ are reconstructed using:

$$\Phi = \mathbf{X}'\tilde{\mathbf{V}}\tilde{\Sigma}^{-1}\mathbf{W}$$

**Proof of equation:**

Assume $\Phi = \mathbf{X}'\tilde{\mathbf{V}}\tilde{\Sigma}^{-1}\mathbf{W}$.

\begin{align*}
\mathbf{A}\Phi &= (\mathbf{X}'\tilde{\mathbf{V}}\tilde{\Sigma}^{-1}\tilde{\mathbf{U}}^*)(\mathbf{X}'\tilde{\mathbf{V}}\tilde{\Sigma}^{-1}\mathbf{W}) \\
&= \mathbf{X}'\tilde{\mathbf{V}}\tilde{\Sigma}^{-1}(\tilde{\mathbf{U}}^*\mathbf{X}'\tilde{\mathbf{V}}\tilde{\Sigma}^{-1})\mathbf{W} \\
&= \mathbf{X}'\tilde{\mathbf{V}}\tilde{\Sigma}^{-1}\tilde{\mathbf{A}}\mathbf{W} \\
&= \mathbf{X}'\tilde{\mathbf{V}}\tilde{\Sigma}^{-1}\mathbf{W}\Lambda \\
&= \Phi \Lambda
\end{align*}

***NOTE:** $(\tilde{\mathbf{A}}, \mathbf{W})$ is a recipe in POD coordinates whereas $(\mathbf{A}, \Lambda)$ is an actual spatial pattern on the physical domain that we can plot. Same eigenvalues, but the eigenvectors live in different spaces, so they have different sizes.*

**Projected Modes:**

In the original paper by Schmid (2010), DMD modes are computed using:

$$\Phi = \tilde{\mathbf{U}} \mathbf{W}$$

However, these modes are not guaranteed exact eigenvectors of $\mathbf{A}$, especially for dynamical systems with high-rank structures.

To find DMD modes corresponding to a zero eigenvalue $\lambda = 0$ (the ones that immediately die out after first time step, no growth $\lambda \gt 0$ or decay $\lambda \lt 0 $), if the exact definition yields $\Phi = \mathbf{X}'\tilde{\mathbf{V}}\tilde{\Sigma}^{-1}\mathbf{W} = 0$, then the projected mode definition should be used.

**Q2a: How to determine the optimal rank $r$ of $\mathbf{X}$ for SVD?**

For an SVD $\mathbf{X} = \tilde{\mathbf{U}}\tilde{\Sigma}\tilde{\mathbf{V}}^*$ with singular values $\sigma_1 \ge \sigma_2 \ge ... \ge 0$, squared singular value $\sigma_i^2$ is representative of energy of $\mathbf{X}$ captured by $i^{\text{th}}$ POD mode.

$$\|\mathbf{X}\|_F^2 = \sum_{i=1}^{m-1} \sigma_i^2 $$

Choosing $r$ means deciding which $\sigma_i$ are real signal and which are noise or negligible.

**Method 1:** Plot $\sigma_i$ against $i$ on a log scale. Real dynamics usually appear as a steep decay followed by a flat plateau (the noise floor). Pick $r$ just before the plateau. This is fast and intuitive, but subjective.

**Method 2:** Choose $r$ based on cumulative energy, such that $E(r) \ge \eta$ where $\eta$ is typically between 0.99 and 0.9999.

$$E(r) = \frac{\sum_{i=1}^{r} \sigma_i^2}{\sum_{i=1}^{m-1} \sigma_i^2}$$

***NOTE:** Energy is not the same as dynamical importance. A weak mode with a small $\sigma_i$ can still be a physically crucial, slowly growing instability, and a high-energy mode can be a dull static mean.*

**Method 3:** Keep $\sigma_i$ if $\sigma_i > \tau$ and discard it otherwise (Optimal hard threshold, Gavish and Donoho, 2014), so that $r$ is simply the number of singular values above $\tau$. $\tau$ is given by:

If we know the noise level $\sigma_{\text{noise}}$,

$$\tau = \lambda(\beta)\,\sqrt{\text{max}(n, m-1)}\;\sigma_{\text{noise}}$$

such that $\lambda(\beta) = \sqrt{2(\beta+1) + \frac{8\beta}{(\beta+1) + \sqrt{\beta^2 + 14\beta + 1}}}$, where $\beta = \frac{\text{min}(n, m-1)}{\text{max}(n, m-1)}$

If we do not know the noise level $\sigma_{\text{noise}}$,

$$\tau = \omega(\beta)\cdot \text{median}(\sigma_1, \dots, \sigma_{m-1})$$

such that $\omega(\beta) = \frac{\lambda(\beta)}{\sqrt{\mu_\beta}} \approx 0.56\,\beta^3 - 0.95\,\beta^2 + 1.82\,\beta + 1.43$ where $\mu_\beta$ is the median of the Marchenko–Pastur distribution.

Even pure noise has singular values, therefore, take a random $m \times n$ matrix that has its singular values spread over a range. The largest of those values can be computed theoretically using the Marchenko–Pastur law. Any singular value of the data inside the noise range cannot be told apart from noise, so it is discarded.

***NOTE:** The theory assumes white noise (independent, identically distributed) noise. Real sensor noise can be correlated or non-uniform, in which case the threshold is only approximate. It also assumes the true signal rank is much smaller than the matrix size, so that the median really reflects noise.*

**Q2b: What is the significance of POD modes in SVD?**

POD modes are column vectors $\mathbf{u} \in \mathbb{R}^n$ that comprise $\tilde{\mathbf{U}}=\begin{bmatrix}\mathbf{u}_1 & \mathbf{u}_2 & ... & \mathbf{u}_r\end{bmatrix} \in \mathbb{C}^{n \times r}$. Each one is a spatial pattern and are order by ordered by $\sigma_i^2$, so truncating $\mathbf{u}_{r+1}, ... , \mathbf{u}_m$ removes least important content by energy. POD does not include temporal information.

Each state $\mathbf{x}$ can be expanded to interpret SVD more clearly:

$$\mathbf{x}(t_k) \approx \sum_{i=1}^{r} \mathbf{u}_i\,\sigma_i\,\overline{v_{k,i}}$$

where $\sigma_i$ is the singular value and $v_{k,i}$ is that mode's amplitude at time $t_k$.

**Q2c: How can $\mathbf{A}$ and $\tilde{\mathbf{A}}$ can have the same non-zero eigenvalues if $\text{dim}(\mathbf{A}) \ne \text{dim}(\tilde{\mathbf{A}})$?**

In trucnated form,

$$\mathbf{A} = \underbrace{\mathbf{X}'\tilde{\mathbf{V}}\tilde{\boldsymbol{\Sigma}}^{-1}}_{(n\times r)}\;\underbrace{\tilde{\mathbf{U}}^*}_{(r\times n)}$$

$\mathbf{X}$ is what we sample. Therefore, during the computation of $\mathbf{A}$, it must be go through the bottleneck of dimension $r$ (for the truncated version using SVD, otherwise, it's $(m-1)$). This means most of the row/column entries of $\mathbf{A}$ is redundant and $\tilde{\mathbf{A}}$ is its equivalent efficient representation.

#### 2.2 Spectral Decomposition

DMD can expand the system state $\mathbf{x}$ in terms of data-driven spectral decomposition.

$$
\mathbf{x}_k = \sum_{j=1}^{r} \boldsymbol{\phi}_j \lambda_j^{k-1} b_j = \boldsymbol{\Phi}\boldsymbol{\Lambda}^{k-1}\mathbf{b} =
\begin{bmatrix}
  \mid & & \mid \\
  \boldsymbol{\phi}_1 & \cdots & \boldsymbol{\phi}_r \\
  \mid & & \mid
\end{bmatrix}
\begin{bmatrix}
  \lambda_1 & & \\
  & \ddots & \\
  & & \lambda_r
\end{bmatrix}^{k-1}
\begin{bmatrix}
  b_1 \\ \vdots \\ b_r
\end{bmatrix}
$$

where $b_j$ is the mode amplitude of $j$-th mode.

The mode amplitude vector $\mathbf{b}$ is generally computed using the first snapshot of data:

$$\mathbf{b} = \Phi^{\dagger}\mathbf{x}_1$$

Since $\Phi \in \mathbb{C}^{n \times r}$, the above computation can be quite expensive. Therefore, it is possible to compute these amplitudes using POD projected data where $\mathbf{W}, \Lambda \in \mathbb{C}^{r \times r}$.

$$\mathbf{b} = (\mathbf{W}\Lambda)^{-1}\tilde{\mathbf{x}}_1$$

**Proof of equation:**

\begin{align*}
\mathbf{x}_1 &= \Phi\mathbf{b} \\
\tilde{\mathbf{U}}\tilde{\mathbf{x}}_1 &= \mathbf{X}'\tilde{\mathbf{V}}\tilde{\Sigma}^{-1}\mathbf{W}\mathbf{b} \\
\tilde{\mathbf{x}}_1 &= \tilde{\mathbf{U}}^*\mathbf{X}'\tilde{\mathbf{V}}\tilde{\Sigma}^{-1}\mathbf{W}\mathbf{b} \\
\tilde{\mathbf{x}}_1 &= \tilde{\mathbf{A}}\mathbf{W}\mathbf{b} \\
\tilde{\mathbf{x}}_1 &= \mathbf{W}\Lambda\mathbf{b} \\
\mathbf{b} &= (\mathbf{W}\Lambda)^{-1}\tilde{\mathbf{x}}_1
\end{align*}

**Data matrix form:**

DMD expansion can be equivalently written as:

$$
\mathbf{x}_k =
\begin{bmatrix}
  \mid & & \mid \\
  \boldsymbol{\phi}_1 & \cdots & \boldsymbol{\phi}_r \\
  \mid & & \mid
\end{bmatrix}
\begin{bmatrix}
  b_1 & & \\
  & \ddots & \\
  & & b_r
\end{bmatrix}
\begin{bmatrix}
  \lambda_1 \\ \vdots \\ \lambda_r
\end{bmatrix}
$$

to express $\mathbf{X}$ as:

$$
\mathbf{X} =
\begin{bmatrix}
  \mid & & \mid \\
  \boldsymbol{\phi}_1 & \cdots & \boldsymbol{\phi}_r \\
  \mid & & \mid
\end{bmatrix}
\begin{bmatrix}
  b_1 & & \\
  & \ddots & \\
  & & b_r
\end{bmatrix}
\begin{bmatrix}
  \lambda_1 & \cdots & \lambda_1^{m-1} \\
  \vdots & \ddots & \vdots \\
  \lambda_r & \cdots & \lambda_r^{m-1}
\end{bmatrix}
$$

**Continous-time form:**

DMD expansion can be written in continous-time domain [Q2d] by introducing continous eigenvalues $\omega = \frac{\log{(\lambda)}}{\Delta t}$

$$\mathbf{x}(t) = \sum_{j=1}^{r} \boldsymbol{\phi}_j e^{\omega_j t} b_j = \Phi \,\exp(\Omega t) \mathbf{b}$$

where $\Omega = \text{diag}(\omega_1, ... , \omega_r)$.

$$
\mathbf{X} \approx
\begin{bmatrix}
  \mid & & \mid \\
  \boldsymbol{\phi}_1 & \cdots & \boldsymbol{\phi}_r \\
  \mid & & \mid
\end{bmatrix}
\begin{bmatrix}
  b_1 & & \\
  & \ddots & \\
  & & b_r
\end{bmatrix}
\begin{bmatrix}
  e^{\omega_1 t_1} & \cdots & e^{\omega_1 t_m} \\
  \vdots & \ddots & \vdots \\
  e^{\omega_r t_1} & \cdots & e^{\omega_r t_m}
\end{bmatrix}
= \boldsymbol{\Phi}\,\mathrm{diag}(\mathbf{b})\,\mathbf{T}(\boldsymbol{\omega})
$$

Thus, this expansions decomposes a state vector or a data matrix into its spatial pattern $\times$ amplitude $\times$ temporal behaviour.

**Q2d: What is the difference between continous- versus discrete-time eigenvalues?**

A discrete-time eigenvalue $\lambda$ quantifies how much the pattern is multiplied by each time step $\Delta t$.

$$\mathbf{x}_{k+1} = \lambda\,\mathbf{x}_k$$

A continuous-time eigenvalue $\omega$ quantifies how fast the pattern grows or shrinks per second.

$$\frac{dx}{dt} = \omega\, x \quad\Rightarrow\quad x(t) = e^{\omega t}x(0)$$

This is useful when comparing different experiments with different time-steps.

### **3. Sparse Identification of Nonlinear Dynamics (SINDy)**

Sparse Identification of Nonlinear Dynamics (SINDy) is a data-driven method used to discover governing equations for dynamical systems purely from measurement data.

This method leverages the fact that $\dot{\mathbf{x}} = \mathbf{f(x)}$ have dynamics $\mathbf{f}$ with only a few active terms in the space of possible righthand side functions (for example, Lorenz equations only have a few linear and quadratic terms per equation). Then, the algorithm seeks to approximate $\mathbf{f(x)}$ by a generalised linear model.

$$\mathbf{f(x)} \approx \sum_{k-1}^{p}\theta_k(\mathbf{x}) \xi_k = \Theta(\mathbf{x})\boldsymbol{\xi}$$

where $\theta_k(\mathbf{x})$ is one candidate function of the state and $\boldsymbol{\Theta}(\mathbf{x})$ is the row of all $p$ candidates: $\Theta(\mathbf{x})=[\theta_1(\mathbf{x}),\ \theta_2(\mathbf{x}),\ \dots,\ \theta_p(\mathbf{x})]$ limited only by our imagination (e.g., $\Theta(\mathbf{x})=[\mathbf{x},\ \mathbf{x}^2,\ \mathbf{x}^3,\ \sin{\mathbf{x}}, \ e^{\mathbf{x}}]$). It is a library of possible linear and nonlinear functions of the governing equations of the system. $\boldsymbol{\xi}$ is the vector of all weights of candidate functions, which is the unknown to be found.

It is then possible to solve for the relevant terms that are active in the dynamics using sparse regression, penalising the number of terms in the dynamics so that $\mathbf{f(x)}$ is approximateed with the fewest non-zero terms in $\boldsymbol{\xi}$ as possible.

For regression, however, we need to several snapshots of state $\mathbf{x}$. Therefore, time-series data is collected and formed into:

$$\mathbf{X} = \left[ \mathbf{x}(t_1) \quad \mathbf{x}(t_2) \quad \cdots \quad \mathbf{x}(t_m) \right]^T$$

A similar matrix of derivatives is formed, either by differentiation of $\mathbf{X}$ or measurement.

$$\dot{\mathbf{X}} = \left[ \mathbf{\dot{x}}(t_1) \quad \mathbf{\dot{x}}(t_2) \quad \cdots \quad \mathbf{\dot{x}}(t_m) \right]^T$$

The dynamical system in may now be represented in terms of the data
matrices.

$$\dot{\mathbf{X}} = \Theta(\mathbf{X}) \, \Xi$$

that is,

$$
\begin{bmatrix}
  \dot{\mathbf{x}}^T(t_1) \\
  \dot{\mathbf{x}}^T(t_2) \\
  \vdots \\
  \dot{\mathbf{x}}^T(t_m)
\end{bmatrix}
=
\left[
\begin{matrix}
  1 \\ 1 \\ \vdots \\ 1
\end{matrix}
\quad
\begin{matrix}
  \mathbf{x}^T(t_1) \\
  \mathbf{x}^T(t_2) \\
  \vdots \\
  \mathbf{x}^T(t_m)
\end{matrix}
\quad
\begin{matrix}
  \mathbf{x}^{2\,T}(t_1) \\
  \mathbf{x}^{2\,T}(t_2) \\
  \vdots \\
  \mathbf{x}^{2\,T}(t_m)
\end{matrix}
\quad \cdots \quad
\begin{matrix}
  \mathbf{x}^{d\,T}(t_1) \\
  \mathbf{x}^{d\,T}(t_2) \\
  \vdots \\
  \mathbf{x}^{d\,T}(t_m)
\end{matrix}
\quad \cdots \quad
\begin{matrix}
  \sin(\mathbf{x}^T(t_1)) \\
  \sin(\mathbf{x}^T(t_2)) \\
  \vdots \\
  \sin(\mathbf{x}^T(t_m))
\end{matrix}
\quad \cdots
\right]
\begin{bmatrix}
  \xi_{1,1} & \xi_{1,2} & \cdots & \xi_{1,n} \\
  \xi_{2,1} & \xi_{2,2} & \cdots & \xi_{2,n} \\
  \vdots & \vdots & \ddots & \vdots \\
  \xi_{p,1} & \xi_{p,2} & \cdots & \xi_{p,n}
\end{bmatrix}
$$

Here, the vector $\mathbf{x}^d$ denotes a dth-degree polynomial vector in the state x. As mentioned earlier, this library of candidate functions $\Theta(\mathbf{X})$ is only limited by one’s imagination.

A parsimonious model will provide an accurate model fit in with as few terms as possible in $\Xi$. Such a model may be identified using a convex sparse regression, such as LASSO, SLTS or $l_1$-regularised sparse regression.



**Discrete-time version:**

For truly discrete-time systems or to avoid numerical differentiation which can amplify noise, discrete SINDy can be used. Assuming the system steps forward as $\mathbf{x}_{k+1} = \mathbf{F(x}_k)$, $\mathbf{F}$ can be approximated with the same library notion:

$$\mathbf{F(x}_k) \approx \Theta(\mathbf{x}_k) \, \Xi$$

For sparse regression, a series of data $\{ \mathbf{x}_1, \mathbf{x}_2, \dots, \mathbf{x}_m \}$ is required.

\begin{align*}
\mathbf{X} &= \left[ \mathbf{x}(t_1) \quad \mathbf{x}(t_2) \quad \cdots \quad \mathbf{x}(t_{m-1}) \right]^T
\\\\
\mathbf{X'} &= \left[ \mathbf{x}(t_2) \quad \mathbf{x}(t_3) \quad \cdots \quad \mathbf{x}(t_m) \right]^T
\end{align*}

Once the data is arranged in above order, dynamical system may then be represented as follows and solved for $\Xi$:

$$
\mathbf{X}' = \Theta(\mathbf{X})\,\Xi
$$

***NOTE:** In continous SINDy, the output is a system of ODEs whereas here, the output is the nonlinear time-step map tied to $\Delta t$. Therefore, when the time-step is changed, the model must be re-fit.*

#### 3.1 Example: Sparse Identification of Lorenz System

The following programmes simulates Lorenz system to get snapshots $\mathbf{X}$ and compute the derivatives $\dot{\mathbf{X}}$. Then, it builds a library $\boldsymbol{\Theta}(\mathbf{X})$ of 20 candidate terms (constant, linear, quadratic, cubic). Solve $\dot{\mathbf{X}}=\boldsymbol{\Theta}\boldsymbol{\Xi}$ with a least-squares fit, then repeat 10 times: zero out small coefficients, then refit using only the surviving terms. The surviving terms are what makes the governing equations as per SINDy.

A sparsification knob $\lambda$ (which can be tuned by the user) is used as a hard-threshold version of L1-regularisation. Any coefficient with $|\xi|<\lambda$ is set to exactly zero and its term is removed from the model.

In [ ]:
import numpy as np
from itertools import combinations_with_replacement
from scipy.integrate import solve_ivp

# ---------- Lorenz system ----------
sigma, beta, rho = 10.0, 8/3, 28.0

def lorenz_deriv(x, t, sigma, beta, rho):
    return [sigma*(x[1] - x[0]),
            x[0]*(rho - x[2]) - x[1],
            x[0]*x[1] - beta*x[2]]

# ---------- Library Theta(X): all monomials up to given order ----------
def poolData(x, n, polyorder):
    m = x.shape[0]
    cols = [np.ones(m)]                                   # constant term
    for order in range(1, polyorder + 1):
        for idx in combinations_with_replacement(range(n), order):
            cols.append(np.prod(x[:, idx], axis=1))       # e.g. x*y, x*x*z
    return np.column_stack(cols)

def libraryNames(n, polyorder, names=("x", "y", "z")):
    out = ["1"]
    for order in range(1, polyorder + 1):
        for idx in combinations_with_replacement(range(n), order):
            out.append("".join(names[i] for i in idx))
    return out

# ---------- Sequential thresholded least squares ----------
def sparsifyDynamics(Theta, dXdt, lamb, n):
    Xi = np.linalg.lstsq(Theta, dXdt, rcond=None)[0]      # initial dense guess
    for k in range(10):
        smallinds = np.abs(Xi) < lamb                     # find small coefficients
        Xi[smallinds] = 0                                 # threshold them
        for ind in range(n):                              # n = state dimension
            biginds = ~smallinds[:, ind]
            # refit using only the surviving terms
            Xi[biginds, ind] = np.linalg.lstsq(
                Theta[:, biginds], dXdt[:, ind], rcond=None)[0]
    return Xi

# ---------- Generate trajectory data ----------
n = 3
x0 = [-8, 8, 27]
dt = 0.001
t = np.arange(0, 100, dt)
sol = solve_ivp(lambda t, x: lorenz_deriv(x, t, sigma, beta, rho),
                (t[0], t[-1]), x0, t_eval=t, rtol=1e-10, atol=1e-10)
x = sol.y.T                                               # shape (m, n)

# ---------- Compute derivatives from the true Lorenz equations ----------
dx = np.zeros_like(x)
for j in range(len(t)):
    dx[j, :] = lorenz_deriv(x[j, :], 0, sigma, beta, rho)

In [ ]:
# ---------- Build library and run sparse regression ----------
Theta = poolData(x, n, 3)                                 # up to 3rd-order polynomials
lamb = 0.025                                              # sparsification knob
Xi = sparsifyDynamics(Theta, dx, lamb, n)

# ---------- Print the discovered equations ----------
names = libraryNames(n, 3)
for k, state in enumerate(["x", "y", "z"]):
    terms = [f"{Xi[j, k]:+.3f}*{names[j]}" for j in range(len(names)) if Xi[j, k] != 0]
    print(f"d{state}/dt =", " ".join(terms))

dx/dt = -10.000*x +10.000*y
dy/dt = +28.000*x -1.000*y -1.000*xz
dz/dt = -2.667*z +1.000*xy


**Q3a: How to decide the optimal $\lambda$ which produces the best model?**

SINDy balances two objectives: prediction error and active number of terms. The idea is to have the best of both. Akaike Information Criterion (AIC) penalises more complex models unless its improved fit justifies the additional parameters.

$$\text{AIC} = 2k - 2 \ln{\hat{(L)}}$$

where $k$ is the number of estimated model parameters and $\hat{L}$ is the likelihood of the model.

#### 3.2 Partial Differential Equation Functional Identification of Nonlinear Dynamics (PDE-FIND)

PDE-FIND is an extension of SINDy modelling framework generalised to include partial derivatives in the library, enabling the identification of partial differential equations.

Consider an example where the library contain spatial derivatives ($u_x$, $u_{xx}$, ...), so the discovered equation can involve space as well as time, although the algorithm is readily generalisable for higher-order PDEs. Take a data field $u(x,t)$ measured at $n$ spatial points and $m$ times.

$$\mathbf{U}=
\begin{bmatrix}
u(x_1,t_1) & u(x_2,t_1) & \cdots & u(x_n,t_1)\\
u(x_1,t_2) & u(x_2,t_2) & \cdots & u(x_n,t_2)\\
\vdots & \vdots & \ddots & \vdots\\
u(x_1,t_m) & u(x_2,t_m) & \cdots & u(x_n,t_m)
\end{bmatrix}$$

Let $\Upsilon\in\mathbb{C}^{mn}$ be the spatial time-series data is arranged into a single column vector.

$$\Upsilon=
\begin{bmatrix}
u(x_1,t_1) & \cdots & u(x_n,t_1) & u(x_1,t_2) & \cdots & u(x_n,t_m)
\end{bmatrix}^T$$

Let $\mathbf{Q}\in\mathbb{C}^{mn}$ be an additional known input that goes into the library as its own column, because the PDE might depend on something other than the field you are solving for. For example, in Schrödinger equation $i\,\psi_t = -\tfrac12\psi_{xx} + V(x)\,\psi$, $\psi(x,t)$ is the unknown field which is in $\Upsilon$ and $V(x)$ is a known potential (tht is set it up or measured) which is in $\mathbf{Q}$.

The candidate linear and nonlinear terms and partial derivatives are then combined into a matrix $\Theta(\Upsilon, \mathbf{Q}) \in \mathbb{C}^{mn\times D}$ (similar to SINDy).

$$
\Theta(\Upsilon,\mathbf{Q})=
\begin{bmatrix}
\mid&\mid&\mid&&\mid&&\mid&\mid&\mid\\
\mathbf{1}&\Upsilon&\Upsilon^2&\cdots&\mathbf{Q}&\cdots&\Upsilon_x&\Upsilon\Upsilon_x&\cdots\\
\mid&\mid&\mid&&\mid&&\mid&\mid&\mid
\end{bmatrix}
$$

***NOTE:** For clean data, finite difference method can be used to compute derivatives. Otherwise for noisy data, polynomial interpolation is required.*

Then, the time derivative $\Upsilon_t$ is also computed and the dynamical system is expressed as:

$$\Upsilon_t = \Theta(\Upsilon,\mathbf{Q}) \, \boldsymbol{\xi}$$

where $\boldsymbol{\xi}$ entries are unknown coefficients corresponding to each term in PDE that needs to evaluated using sparsity-promoting regression, as in SINDy.

**Q3b: What is the optimal regressor for PDE-FIND?**

Unlike SINDy, PDE libraries have highly correlated columns ($u$, $u^2$, $u^3$ look alike), which makes L1-regularised least squares ill-conditioned. Instead, sequential threshold ridge regression (STRidge in Algorithm 1) is recommended.

For a given tolerance and threshold $\lambda$, this gives a sparse approximation to $\boldsymbol{\xi}$. The tolerance of Algorithm 1 is iteratively refined to find the best predictor based on the selection criteria:

$$\hat{\boldsymbol{\xi}} = \operatorname{argmin}_{\boldsymbol{\xi}} \Vert{}\Theta(\Upsilon, \mathbf{Q})\boldsymbol{\xi} - \Upsilon_t\Vert{}_2^2 + \epsilon \kappa(\Theta(\Upsilon, \mathbf{Q})) \Vert{}\boldsymbol{\xi}\Vert{}_0$$

where $\kappa(\Theta)$ is the condition number which makes the sparsity penalty stronger when the library is badly conditioned and penalising $\Vert{}\boldsymbol{\xi}\Vert{}_0$ discourages overfitting.

**Q3c: What is the effect of training data diversity on performance?**

It is necessary to use sufficiently diverse training data to correctly identify the governing equations because different governing equations can explain the same observed behaviour.

***NOTE:** PDE-FIND algorithm can also be used to identify PDEs based on Lagrangian measurements that follow the path of individual particles rather than measuring the entire spatial field.*

**Example: Korteweg–de Vries (KdV) equation**

$$u_t+6uu_x+u_{xxx}=0$$

If our measurement system only observe one soliton (A localised wave that maintains its shape and amplitude as it propagates, and survives interactions with other solitons.), which travels at a constant speed. The algorithm can explain its motion using the much simpler linear advection equation $u_t+cu_x=0$. In reality, linear advection equation is a linear approximation under specific conditions.

If the measurement system observe two waves with different amplitudes, the larger soliton travels faster than the smaller one given that the system does not fall into linearisable conditions. Then, $u_t + cu_x = 0$ is no longer valid. Thus, the algorithm needs nonlinear and dispersive terms to explain the observations, allowing it to identify the KdV equation.

In [ ]:
import numpy as np
import plotly.graph_objects as go

x = np.linspace(-30, 30, 500)
t = np.linspace(0, 12, 80)
k1, k2 = np.sqrt(2), np.sqrt(0.8)
B = ((k1-k2)/(k1+k2))**2

# Exact KdV two-soliton solution
def kdv(x, t):
    a, b = k1*(x-k1**2*t+20), k2*(x-k2**2*t+10)
    z = np.array([np.zeros_like(x), a, b, a+b+np.log(B)])
    w = np.exp(z-z.max(axis=0))
    w /= w.sum(axis=0)
    p = np.array([0, k1, k2, k1+k2])[:, None]
    return 2*((w*p**2).sum(axis=0) - (w*p).sum(axis=0)**2)

# Both start from exactly the same initial profile
def waves(t):
    return kdv(x-1.4*t, 0), kdv(x, t)

fig = go.Figure()
for y, name in zip(waves(0), ["Linear advection", "KdV"]):
    fig.add_scatter(x=x, y=y, name=name, mode="lines")

fig.frames = [
    go.Frame(data=[go.Scatter(y=y) for y in waves(ti)], name=str(i))
    for i, ti in enumerate(t)]

fig.update_layout(
    xaxis_range=[-30, 30], yaxis_range=[0, 1.2],
    updatemenus=[dict(type="buttons", buttons=[
        dict(label="Play", method="animate",
             args=[None, {"frame": {"duration": 30, "redraw": False}}]),
        dict(label="Pause", method="animate",
             args=[[None], {"mode": "immediate",
                            "frame": {"duration": 0}}])])])
fig.show()

#### 3.3 SINDy for Rational Function Nonlinearities

Due to seperation of timescales, rational function nonlinearities can arise in dynamical systems, which are not sparse linear combinations of a few basis functions. For numberator and denomenator polynomials $f_N(\mathbf{x})$ and $f_D(\mathbf{x})$,

$$\dot{\mathbf{x}} =
\begin{bmatrix}
\frac{f_{N,1}(\mathbf{x})}{f_{D,1}(\mathbf{x})} \\[6pt]
\vdots \\[6pt]
\frac{f_{N,n}(\mathbf{x})}{f_{D,n}(\mathbf{x})}
\end{bmatrix}$$

It is necessary to reformulate the dynamics in an implicit ordinary differential equation. For $k$-th variable,

$$\dot{x}_k = \frac{f_N(\mathbf{x})}{f_D(\mathbf{x})}$$

$$f_N(\mathbf{x}) - f_D(\mathbf{x})\,\dot{x}_k=0$$

Function library $\Theta$ is also generalised in terms of $\mathbf{x}$ and $\dot{x}_k$.

$$\Theta(\mathbf{X}, \dot{x}_k(\mathbf{t})) = \begin{bmatrix}
\Theta_N(\mathbf{X}) & \text{diag}(\dot{x}_k(\mathbf{t}))\Theta_D(\mathbf{X})
\end{bmatrix}$$

where $\text{diag}(\dot{x}_k(\mathbf{t}))\Theta_D(\mathbf{X}) = \begin{bmatrix}
\dot{x}_k(\mathbf{t}) & (\dot{x}_k x_k)(\mathbf{t}) & (\dot{x}_k x_k^2)(\mathbf{t}) & \cdots
\end{bmatrix}$

***NOTE:** In most cases, $\Theta_N(\mathbf{X}) = \Theta_D(\mathbf{X})$, so the augmented library is only twice the size of original library*.

Then, the dynamical system can be represented as:

$$\Theta(\mathbf{X}, \dot{x}_k(\mathbf{t})) \boldsymbol{\xi}_k = \boldsymbol{0}$$

Note that sparsest possible solution for this system is $\boldsymbol{\xi}_k = 0$. Next option would be to search the null space $\Theta \boldsymbol{\xi}_k = \boldsymbol{0}$ for the sparsest non-zero vector $\boldsymbol{\xi}_k$. However, measurement noise makes the numerical null space difficult to determine reliably.

Insteaad, one coefficient can be fixed (subject one term in the library) to remove the arbitrary scaling ambiguity and prevent the all-zero solution. Then, the homogeneous problem $\Theta \boldsymbol{\xi}_k = 0$ becomes an ordinary regression problem.

$$\sum_{j=1}^{p}\boldsymbol{\theta}_j\xi_j=\mathbf{0}$$

$$\boldsymbol{\theta}_q = -\sum_{j=1 \\ j \ne q}^{p}\boldsymbol{\theta}_j\xi_j$$

For example, instead of solving $\begin{bmatrix}\dot{x}&x\dot{x}&x^2\end{bmatrix}\xi=0$, one candidate term is subjected $\dot{x} = x^2 - x \dot{x}$ and ordinary sparse regression is performed. This process is repeated with different candidate terms until a sparse model that accurately describes the data is found.

**Q3d: Can implicit SINDy identify equations more complicated than rational nonlinearities?**

Instead of candidate libraries with only states $\mathbf{X}$, consider a library that includes its derivatives as well:

$$\Theta([\mathbf{X},\mathbf{\dot{X}},\mathbf{\ddot{X}}]) = \begin{bmatrix}
\mathbf{X} & \mathbf{X}^2 & \cdots & \mathbf{X}\mathbf{\dot{X}} & \mathbf{\dot{X}}\mathbf{\ddot{X}} & \cdots & \mathbf{\dot{X}}^p\mathbf{\ddot{X}}^q
\end{bmatrix}$$

Idenitification of sparsest coefficient vector for such libraries provides more flexibility in identifying nonlinear equations with mixed terms containing various powers of any combination of derivatives and states, including rational nonlinearities.

### **4. Koopman Operator Theory**

Koopman operator theory is a mathematical framework that transforms nonlinear dynamical systems into a linear (though infinite-dimensional) representation by shifting the focus from system states $\mathbf{x}$ to measurement functions $\mathbf{y} = g(\mathbf{x})$ (observables). The functions $g$ are elements of an infinite-dimensional Hilbert space $\mathcal{H}$ of Lebesgue square-integrable functions [Q4a].

$$g:\mathbb{R}^n \rightarrow \mathbb{R}, \quad g \in \mathcal{H}$$

The Koopman operator $\mathcal{K}_t$ is an infinite-dimensional linear operator that advances the observable one time step forward. The operator $\mathcal{K}$ is linear eventhough the dynamics $\mathbf{F}$ are nonlinear in $\mathbf{x}_{k+1} = \mathbf{F}(\mathbf{x}_k)$.

$$\mathcal{K}_t\,\mathbf{y}_k = \mathcal{K}_t\,g(\mathbf{x}_k) = g(\mathbf{F}_t(\mathbf{x}_k)) = g(\mathbf{x}_{k+1}) = \mathbf{y}_{k+1}$$

For a discrete-time system with time step $\Delta t$,

$$\mathbf{y}_{k+1} = \mathcal{K}_{\Delta t}\,\mathbf{y}_k$$

The objective is to obtain a finite-dimensional matrix approximation (a Koopman-invariant subspace) of that infinite-dimensional Koopman operator in order to analyse, predict and control the nonlinear dynamics using the linear techniques available.

Let $\{y_k\}_{k=1}^{\infty}$ be a basis of $\mathcal{H}$. Any observable can be written as:

$$g = \sum_{k=1}^{\infty} \alpha_k y_k \approx \alpha_1 y_{s_1} + \alpha_2 y_{s_2} + \cdots + \alpha_m y_{s_m}$$

***NOTE:** Observables are taken from the Hilbert space of Lebesgue square-integrable functions, $\int_{\mathbb{R}^n}|g(\mathbf{x})|^2\,d\mathbf{x}<\infty$, because its inner product $\langle g_1,g_2\rangle=\int_{\mathbb{R}^n} g_1\overline{g_2}\,d\mathbf{x}$ and completeness allow any observable to be expanded in a basis, $g=\sum_k\alpha_k y_k$, with coefficients found by projection, a quantifiable truncation error, and guaranteed convergence of the infinite sum.*

If $\mathcal{S}=\text{span}\{y_{s_1},\dots,y_{s_m}\}$ is Koopman-invariant ($\mathcal{K}\mathcal{S}\subseteq\mathcal{S}$), then:

$$\mathcal{K}g = \beta_1 y_{s_1} + \beta_2 y_{s_2} + \cdots + \beta_m y_{s_m}$$

The relationship between coefficients that evolve in time (dynamics) is given by:

$$\boldsymbol{\beta}=\mathbf{K}\boldsymbol{\alpha}$$

where $\mathbf{K}\in\mathbb{R}^{m\times m}$ is the finite-dimensional matrix representation of $\mathcal{K}$ on $\mathcal{S}$.